# Lecture 6 — Backpropagation

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 6.  
**Merged with:** Stanford CS231n 2026 [Lecture 4: *Neural Networks and Backpropagation*](https://cs231n.stanford.edu/slides/2026/lecture_4.pdf) (second half) and its backprop discussion section.

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture; the CS231n 2026 slides were read in full. **[CS231n 2026]** marks content only there; **🔎** marks material beyond both.

### How to read this notebook
* Each **code cell does one thing**; explanations come before it and **"What to notice"** after important results.
* Every function and class says **what it computes, what goes in, and what comes out**.
* Every example from the slides is **reproduced number by number**. **✍️ Try it** boxes suggest experiments. Run cells **in order**.

**The problem: how do we compute gradients?** For a network like $s = W_2\max(0, W_1x)$ with a hinge loss and regularization,
$$L = \sum_i\sum_{j\neq y_i}\max(0, s_j - s_{y_i} + 1) + \lambda R(W_1) + \lambda R(W_2)$$
if we can compute $\partial L/\partial W_1$ and $\partial L/\partial W_2$, we can learn them.

**(Bad) idea: derive the gradient on paper.** Very tedious (lots of matrix calculus). Not modular: switch from hinge to softmax and you **re-derive everything from scratch**. Not feasible for very complex models (AlexNet; a Neural Turing Machine unrolled over time).

**Better idea: computational graphs + backpropagation.** Break the computation into simple operations ("nodes"), each of which knows its own **local** derivative; the chain rule then combines them mechanically, in one backward sweep.

| § | Topic | What you'll build |
|---|---|---|
| 1 | A simple example: $(x + y)z$ | Forward and backward by hand |
| 2 | Upstream × local = downstream; a sigmoid neuron | Every number on the slides' graph |
| 3 | Patterns: add, multiply, copy, max | All four with the slides' numbers |
| 4 | Implementations | "Flat" code; a forward/backward API; a tiny **autograd engine** |
| 5 | Backprop with **vectors** | The ReLU example; why we never build Jacobians |
| 6 | Backprop with **matrices**: $y = xw$ | The slides' example (and a typo in it) |
| 7 | A network from modular layers | Linear / ReLU / softmax layers, gradient-checked, trained |
| 8 | 🔎 Forward vs. reverse mode | Dual numbers; why backprop is the right direction |
| 9 | Higher-order derivatives | Hessian-vector products |
| 10 | 🔎 The memory cost of backprop | Activations stored for the backward pass |
| 11 | PyTorch autograd | (Colab) |

### Setup

In [ ]:
import numpy as np                          # arrays and math
import matplotlib.pyplot as plt             # plots
from sklearn.datasets import load_digits    # digits, for training a network in Sec. 7
import math, time

np.set_printoptions(precision=4, suppress=True)

---
## 1. A simple example

$$f(x, y, z) = (x + y)\,z\qquad\text{with } x = -2,\ y = 5,\ z = -4$$
**The graph:** an add node computes $q = x + y$, then a multiply node computes $f = q\cdot z$.

**1. Forward pass:** compute the outputs, left to right.

In [ ]:
x, y, z = -2.0, 5.0, -4.0
q = x + y          # add node
f = q * z          # multiply node
print(f"q = x + y = {q},   f = q * z = {f}")

**2. Backward pass:** compute derivatives, right to left. Each node's **local derivatives** are simple:
* add node $q = x + y$: $\ \partial q/\partial x = 1$, $\ \partial q/\partial y = 1$
* multiply node $f = qz$: $\ \partial f/\partial q = z$, $\ \partial f/\partial z = q$

Start from $\partial f/\partial f = 1$ and work back. To get $\partial f/\partial x$ we need the **chain rule**: $x$ affects $f$ only through $q$, so
$$\frac{\partial f}{\partial x} = \underbrace{\frac{\partial f}{\partial q}}_{\text{upstream gradient}}\cdot\underbrace{\frac{\partial q}{\partial x}}_{\text{local gradient}}$$

In [ ]:
df_df = 1.0                    # base case
df_dz = q * df_df              # multiply node: local gradient wrt z is q       ->  3
df_dq = z * df_df              # multiply node: local gradient wrt q is z       -> -4
df_dx = 1.0 * df_dq            # add node: local gradient 1, times upstream -4  -> -4
df_dy = 1.0 * df_dq            #                                                -> -4
print(f"df/dx = {df_dx},  df/dy = {df_dy},  df/dz = {df_dz}")

**Check numerically**, by nudging each input a tiny bit:

In [ ]:
fn = lambda x_, y_, z_: (x_ + y_) * z_
h = 1e-6
print("numerical:", [round((fn(x + h, y, z) - f) / h, 4), round((fn(x, y + h, z) - f) / h, 4), round((fn(x, y, z + h) - f) / h, 4)])

---
## 2. Upstream × local = downstream

Every node in any graph does the same three things:
* **Forward:** receive inputs $x, y$, compute output $z$.
* **Backward:** receive the **upstream gradient** $\frac{\partial L}{\partial z}$ (how much the final loss $L$ changes when this node's output changes), multiply by the **local gradients** $\frac{\partial z}{\partial x}$, $\frac{\partial z}{\partial y}$ (which depend only on this node), and send the **downstream gradients** to the inputs:
$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z}\,\frac{\partial z}{\partial x},\qquad\frac{\partial L}{\partial y} = \frac{\partial L}{\partial z}\,\frac{\partial z}{\partial y}$$

Each node only needs **local** information. That's why backprop is **modular**: change the loss, and no other node needs re-deriving.

### 2.1 Another example: a sigmoid neuron
$$f(w, x) = \frac{1}{1 + e^{-(w_0x_0 + w_1x_1 + w_2)}},\qquad w_0 = 2,\ x_0 = -1,\ w_1 = -3,\ x_1 = -2,\ w_2 = -3$$
Built from primitive nodes: two multiplies, two adds, a negation ($\times-1$), $\exp$, $+1$, and $1/x$. Their local derivatives:

| Node | Local derivative |
|---|---|
| $e^x$ | $e^x$ |
| $1/x$ | $-1/x^2$ |
| $ax$ (multiply by a constant) | $a$ |
| $c + x$ (add a constant) | $1$ |

**Forward pass**, one node at a time:

In [ ]:
w0, x0, w1, x1, w2 = 2.0, -1.0, -3.0, -2.0, -3.0
a = w0 * x0          # -2
b = w1 * x1          #  6
c = a + b            #  4
d = c + w2           #  1
e = -1 * d           # -1
g = math.exp(e)      #  0.37
h_ = 1 + g           #  1.37
out = 1 / h_         #  0.73
print(f"w0*x0 = {a}, w1*x1 = {b}, sum = {c}, +w2 = {d}, *-1 = {e}, exp = {g:.2f}, +1 = {h_:.2f}, 1/x = {out:.2f}")

**Backward pass**, from the output back, multiplying upstream by local at every node:

In [ ]:
d_out = 1.0                          # base case
d_h = d_out * (-1 / h_ ** 2)         # 1/x  : local -1/x^2       -> -0.53
d_g = d_h * 1                        # +1   : local 1            -> -0.53
d_e = d_g * math.exp(e)              # exp  : local e^x          -> -0.20
d_d = d_e * (-1)                     # *-1  : local -1           ->  0.20
d_c, d_w2 = d_d * 1, d_d * 1         # add  : local 1, to both   ->  0.20, 0.20
d_a, d_b = d_c * 1, d_c * 1          # add  : local 1, to both   ->  0.20, 0.20
d_w0, d_x0 = d_a * x0, d_a * w0      # mul  : swap the inputs    -> -0.20, 0.39
d_w1, d_x1 = d_b * x1, d_b * w1      # mul  : swap the inputs    -> -0.39, -0.59
print(f"backward: 1/x {d_h:.2f}, +1 {d_g:.2f}, exp {d_e:.2f}, *-1 {d_d:.2f}, add {d_c:.2f}")
print(f"gradients: w0 {d_w0:.2f}, x0 {d_x0:.2f}, w1 {d_w1:.2f}, x1 {d_x1:.2f}, w2 {d_w2:.2f}")
print("slides: [0.2] x [1] = 0.2 at the adds;  w0: [0.2] x [-1] = -0.2;  x0: [0.2] x [2] = 0.4")

**A shortcut.** *The graph for a function isn't unique: choose one where each node's local gradient is easy to express!* The last four nodes together compute the **sigmoid** $\sigma(x) = \frac{1}{1 + e^{-x}}$, whose derivative simplifies nicely:
$$\frac{d\sigma}{dx} = \frac{e^{-x}}{(1 + e^{-x})^2} = \left(\frac{1 + e^{-x} - 1}{1 + e^{-x}}\right)\left(\frac{1}{1 + e^{-x}}\right) = (1 - \sigma(x))\,\sigma(x)$$
So one "sigmoid node" replaces four: upstream $1.00\ \times$ local $(1 - 0.73)(0.73) = 0.2$:

In [ ]:
sigma = 1 / (1 + math.exp(-d))
print(f"one sigmoid node: (1 - {sigma:.2f}) * {sigma:.2f} = {(1 - sigma) * sigma:.4f}   (the four separate nodes gave {d_d:.4f})")

---
## 3. Patterns in gradient flow

| Node | Forward | What happens to the gradient | Slides' example |
|---|---|---|---|
| **add**: a **distributor** | $z = x + y$ | both inputs get the upstream gradient unchanged | 3 + 4 = 7; upstream 2 → **2, 2** |
| **multiply**: a **"swap multiplier"** | $z = xy$ | each input gets upstream × the **other** input | 2 × 3 = 6; upstream 5 → **5·3 = 15, 5·2 = 10** |
| **copy**: an **adder** | one value used twice | the upstream gradients **add** | 7 used twice; upstreams 4, 2 → **6** |
| **max**: a **router** | $z = \max(x, y)$ | the larger input gets everything, the other 0 | max(4, 5) = 5; upstream 9 → **0, 9** |

Each as a tiny function returning (forward output, gradients for the inputs):

In [ ]:
def add_node(x, y, upstream):  return x + y, (upstream, upstream)
def mul_node(x, y, upstream):  return x * y, (upstream * y, upstream * x)
def copy_node(x, upstreams):   return (x, x), sum(upstreams)
def max_node(x, y, upstream):  return max(x, y), ((upstream, 0) if x > y else (0, upstream))

print("add  3 + 4,     upstream 2    ->", add_node(3, 4, 2))
print("mul  2 * 3,     upstream 5    ->", mul_node(2, 3, 5))
print("copy 7,         upstreams 4, 2 ->", copy_node(7, (4, 2)))
print("max  max(4, 5), upstream 9    ->", max_node(4, 5, 9))

**What to notice:** the **copy** rule is why gradients **accumulate** (`+=`) whenever a value is used in several places, such as a filter shared across image positions (Lecture 7) or weights shared across time steps (Lecture 12).

---
## 4. Implementing backprop

### 4.1 "Flat" code
Write the forward pass as a list of statements, then the backward pass as the **same statements in reverse**, one gradient line per forward line (the slides' version of the sigmoid neuron):

In [ ]:
def sigmoid_neuron_flat(w0, x0, w1, x1, w2):
    # Forward and backward pass of the sigmoid neuron, written out line by line. Returns (output, gradients).
    s0 = w0 * x0                         # forward pass
    s1 = w1 * x1
    s2 = s0 + s1
    s3 = s2 + w2
    L = 1 / (1 + math.exp(-s3))          # sigmoid

    grad_L = 1.0                         # backward pass (base case)
    grad_s3 = grad_L * (1 - L) * L       # sigmoid
    grad_w2 = grad_s3                    # add
    grad_s2 = grad_s3
    grad_s0 = grad_s2                    # add
    grad_s1 = grad_s2
    grad_w1 = grad_s1 * x1               # multiply
    grad_x1 = grad_s1 * w1
    grad_w0 = grad_s0 * x0               # multiply
    grad_x0 = grad_s0 * w0
    return L, (grad_w0, grad_x0, grad_w1, grad_x1, grad_w2)

L, grads = sigmoid_neuron_flat(2.0, -1.0, -3.0, -2.0, -3.0)
print(f"output {L:.4f}; gradients (w0, x0, w1, x1, w2) = {np.round(grads, 4)}")

This is exactly how you'll write layers' backward passes in the assignments.

### 4.2 A modular forward / backward API
Real frameworks make each operation an **object**: `forward` computes the output and **saves whatever `backward` will need**; `backward` multiplies the upstream gradient by the local gradients. (The slides show PyTorch's real code for a multiply node with `ctx.save_for_backward`, and its sigmoid layer, whose backward is literally `grad_output * (1 - output) * output`.)

In [ ]:
class Multiply:
    # z = x * y
    def forward(self, x, y):
        self.x, self.y = x, y                  # save the inputs for the backward pass
        return x * y
    def backward(self, grad_z):
        return self.y * grad_z, self.x * grad_z   # (dL/dx, dL/dy): swap the inputs

class Sigmoid:
    # z = 1 / (1 + exp(-x))
    def forward(self, x):
        self.out = 1 / (1 + math.exp(-x))      # save the OUTPUT: the derivative needs only it
        return self.out
    def backward(self, grad_out):
        return grad_out * (1 - self.out) * self.out

m = Multiply(); print("Multiply: forward(-2, 3) =", m.forward(-2.0, 3.0), "; backward(1.5) =", m.backward(1.5))
sg = Sigmoid(); sg.forward(1.0); print(f"Sigmoid: backward at x = 1 with upstream 1 -> {sg.backward(1.0):.4f}")

### 4.3 🔎 A tiny autograd engine
Combine the two ideas into an **automatic** system, like PyTorch's autograd but for single numbers (in the style of Karpathy's *micrograd*):
1. Each `Value` stores a number (`data`), its gradient (`grad`), the `Value`s it was computed from (`parents`), and a small function that applies **its** local backward rule.
2. Every operation (`+`, `*`, `exp`, …) creates a new `Value` and attaches the right backward rule.
3. `backward()` sorts the graph so every node comes **after** everything that depends on it (a **topological order**), sets the output's gradient to 1, and runs each node's rule in that order, **accumulating with `+=`** (the copy rule).

**Step 1: the class**, with addition and multiplication:

In [ ]:
class Value:
    # A scalar that remembers how it was computed, so gradients can flow back through it.
    def __init__(self, data, parents=(), op=""):
        self.data, self.grad = float(data), 0.0
        self._parents, self._op = parents, op
        self._backward = lambda: None              # this node's local backward rule (set by each operation)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():                            # add: pass the upstream gradient to both inputs
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():                            # multiply: each input gets upstream x the other input
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

**Step 2: more operations.** We add them to the class after the fact (Python allows that), each with its local derivative:

In [ ]:
def _exp(self):
    out = Value(math.exp(self.data), (self,), "exp")
    def _backward(): self.grad += out.data * out.grad              # d/dx e^x = e^x
    out._backward = _backward
    return out

def _pow(self, k):
    out = Value(self.data ** k, (self,), f"**{k}")
    def _backward(): self.grad += k * self.data ** (k - 1) * out.grad   # d/dx x^k = k x^(k-1)
    out._backward = _backward
    return out

def _relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward(): self.grad += (self.data > 0) * out.grad        # router: pass only if the input was positive
    out._backward = _backward
    return out

Value.exp, Value.__pow__, Value.relu = _exp, _pow, _relu
Value.__radd__ = Value.__add__                                       # so that 1 + Value works
Value.__rmul__ = Value.__mul__                                       # so that 2 * Value works
Value.__neg__ = lambda self: self * -1
Value.__sub__ = lambda self, other: self + (-other)
Value.__truediv__ = lambda self, other: self * other ** -1
Value.__rtruediv__ = lambda self, other: Value(other) * self ** -1   # so that 1 / Value works

**Step 3: `backward()`**: topological sort (depth-first: visit all parents before adding a node), then run the rules from the output back.

In [ ]:
def _backward_pass(self):
    # Compute d(self)/d(every Value it depends on), storing each result in that Value's .grad.
    order, seen = [], set()
    def visit(v):
        if v not in seen:
            seen.add(v)
            for p in v._parents:
                visit(p)
            order.append(v)                 # v is added only after everything it depends on
    visit(self)
    self.grad = 1.0                         # base case: d(self)/d(self) = 1
    for v in reversed(order):               # from the output back to the inputs
        v._backward()

Value.backward = _backward_pass

**Step 4: test it on both slide examples**, plus a value used twice (the copy rule):

In [ ]:
x, y, z = Value(-2), Value(5), Value(-4)
f = (x + y) * z; f.backward()
print(f"(x + y) z = {f.data}:  grads x {x.grad}, y {y.grad}, z {z.grad}")

W0, X0, W1, X1, W2 = (Value(v) for v in (2, -1, -3, -2, -3))
out = 1 / (1 + (-(W0 * X0 + W1 * X1 + W2)).exp()); out.backward()
print(f"sigmoid neuron = {out.data:.4f}:  grads w0 {W0.grad:.4f}, x0 {X0.grad:.4f}, w1 {W1.grad:.4f}, x1 {X1.grad:.4f}, w2 {W2.grad:.4f}")

xv = Value(3.0); fv = xv * xv + xv; fv.backward()
print(f"x*x + x at x = 3: df/dx = {xv.grad}  (by hand: 2x + 1 = 7; x is used three times, so its gradients add)")

**Step 5: a small neural network in the engine**, checked against numerical gradients:

In [ ]:
net_rng = np.random.default_rng(1)
W1_np, W2_np, x_np = net_rng.normal(size=(3, 4)), net_rng.normal(size=(4,)), net_rng.normal(size=3)

def net_numpy(W1_, W2_):
    # The same network in plain NumPy: squared error of a 3-4-1 ReLU network, target 1.
    h = np.maximum(0, x_np @ W1_); return float((h @ W2_ - 1.0) ** 2)

W1v = [[Value(W1_np[i, j]) for j in range(4)] for i in range(3)]
W2v = [Value(v) for v in W2_np]
hidden = [sum((Value(x_np[i]) * W1v[i][j] for i in range(3)), Value(0)).relu() for j in range(4)]
loss = (sum((hidden[j] * W2v[j] for j in range(4)), Value(0)) - 1.0) ** 2
loss.backward()

numerical = np.zeros((3, 4))
for i in range(3):
    for j in range(4):
        Wp, Wm = W1_np.copy(), W1_np.copy(); Wp[i, j] += 1e-6; Wm[i, j] -= 1e-6
        numerical[i, j] = (net_numpy(Wp, W2_np) - net_numpy(Wm, W2_np)) / 2e-6
engine = np.array([[W1v[i][j].grad for j in range(4)] for i in range(3)])
print(f"loss: engine {loss.data:.6f}, numpy {net_numpy(W1_np, W2_np):.6f};  max |engine gradient - numerical| = {np.abs(engine - numerical).max():.1e}")

**What to notice:** about 40 lines of code compute exact gradients for **any** expression built from these operations. PyTorch's autograd is the same algorithm, for tensors instead of single numbers.

---
## 5. Backprop with vectors

**Recap: derivatives of different shapes.**
* **Scalar → scalar** ($x\in\mathbb{R}\to y\in\mathbb{R}$): the ordinary **derivative**. *If $x$ changes a little, how much does $y$ change?*
* **Vector → scalar** ($x\in\mathbb{R}^N\to y$): the **gradient** $\frac{\partial y}{\partial x}\in\mathbb{R}^N$. *For each element of $x$, how much does $y$ change?*
* **Vector → vector** ($x\in\mathbb{R}^N\to y\in\mathbb{R}^M$): the **Jacobian** $\frac{\partial y}{\partial x}\in\mathbb{R}^{N\times M}$, entry $(n, m) = \partial y_m/\partial x_n$. *For each element of $x$, how much does each element of $y$ change?*

**In a network,** a node takes vectors $x, y$ and outputs a vector $z$, **but the loss $L$ is still a scalar**. The upstream gradient $\frac{\partial L}{\partial z}$ is a vector (one number per element of $z$); the local gradients are **Jacobians**; and the downstream gradient is a **matrix-vector product**:
$$\frac{\partial L}{\partial x} = \frac{\partial z}{\partial x}\,\frac{\partial L}{\partial z}$$
**The gradient with respect to a variable always has the same shape as the variable.**

### 5.1 Example: elementwise ReLU, $z = \max(0, x)$
The slides' numbers: $x = [1, -2, 3, -1]$ and upstream $\frac{\partial L}{\partial z} = [4, -1, 5, 9]$. Each $z_i$ depends only on $x_i$, with derivative 1 if $x_i > 0$, else 0, so the Jacobian is **diagonal**:

In [ ]:
x4 = np.array([1.0, -2.0, 3.0, -1.0])
z4 = np.maximum(0, x4)
dL_dz = np.array([4.0, -1.0, 5.0, 9.0])                    # upstream gradient
J = np.diag((x4 > 0).astype(float))                        # Jacobian dz/dx: 4 x 4, diagonal
print("x =", x4, "  z = max(0, x) =", z4)
print("Jacobian dz/dx:\n", J)
print("dL/dx = J @ dL/dz =", J @ dL_dz, "  (slides: [4, 0, 5, 0])")

**The Jacobian is sparse: its off-diagonal entries are always zero! Never build it explicitly; use the implicit rule instead:** $\left(\frac{\partial L}{\partial x}\right)_i = \left(\frac{\partial L}{\partial z}\right)_i$ if $x_i > 0$, else 0. Same result, no 4×4 matrix:

In [ ]:
print("implicit: where(x > 0, dL/dz, 0) =", np.where(x4 > 0, dL_dz, 0.0))

---
## 6. Backprop with matrices (or tensors)

Now $x$ is a matrix ($D_x\times M_x$). The local "Jacobians" become 4-dimensional, $[(D_x\times M_x)\times(D_z\times M_z)]$: *for each element of $x$, how much does it affect each element of $z$?* And as always, **$\frac{\partial L}{\partial x}$ has the same shape as $x$.**

### 6.1 Example: matrix multiply, $y = xw$
The slides' example: $x$ is $[N\times D] = 2\times3$, $w$ is $[D\times M] = 3\times4$, so $y$ is $2\times4$, with a given upstream gradient $\frac{\partial L}{\partial y}$ ($2\times4$).

**How big would the Jacobians be in a real network?** $\frac{\partial y}{\partial x}$ is $[(N\times D)\times(N\times M)]$. With $N = 64$, $D = M = 4096$:

In [ ]:
N_, D_, M_ = 64, 4096, 4096
print(f"dy/dx would have ({N_ * D_:,} x {N_ * M_:,}) entries = {N_ * D_ * N_ * M_ * 4 / 2 ** 30:.0f} GiB in float32")

**~256 GB for one Jacobian of one layer! We must work with them implicitly.** First, the slides' matrices:

In [ ]:
x = np.array([[2, 1, -3], [-3, 4, 2]], float)
w = np.array([[3, 2, 1, -1], [2, 1, 3, 2], [3, 2, 1, -2]], float)
dL_dy = np.array([[2, 3, -3, 9], [-8, 1, 4, 6]], float)
y = x @ w
print("y = x w =\n", y)
print("by hand, y[0, 0] = 2*3 + 1*2 + (-3)*3 =", 2 * 3 + 1 * 2 + (-3) * 3)
print("the slides print y = [[13, 9, -2, -6], [5, 2, 17, 1]]  <- does not equal x w")

🔎 **The $y$ on the slide is not $xw$** (a typo in the example). It doesn't affect the gradients, which only need $x$, $w$, and $\frac{\partial L}{\partial y}$.

**Deriving $\frac{\partial L}{\partial x}$, one element at a time:**
* **Q: Which parts of $y$ does one element $x_{n,d}$ affect?** The **whole row** $y_{n,\cdot}$ (since $y_{n,m} = \sum_dx_{n,d}w_{d,m}$).
* **Q: How much does $x_{n,d}$ affect $y_{n,m}$?** $\frac{\partial y_{n,m}}{\partial x_{n,d}} = w_{d,m}$.
* **So, by the chain rule** (summing over every $y$ it affects):
$$\frac{\partial L}{\partial x_{n,d}} = \sum_m\frac{\partial L}{\partial y_{n,m}}\,w_{d,m}\quad\Longrightarrow\quad\underbrace{\frac{\partial L}{\partial x}}_{[N\times D]} = \underbrace{\frac{\partial L}{\partial y}}_{[N\times M]}\underbrace{w^\top}_{[M\times D]}$$
* **By similar logic:** $\ \underbrace{\frac{\partial L}{\partial w}}_{[D\times M]} = \underbrace{x^\top}_{[D\times N]}\underbrace{\frac{\partial L}{\partial y}}_{[N\times M]}$

**These formulas are easy to remember: they're the only way to make the shapes match up!** First, one element by hand, $\frac{\partial L}{\partial x_{0,0}} = \sum_m\frac{\partial L}{\partial y_{0,m}}w_{0,m}$:

In [ ]:
terms = " + ".join(f"({a:g})({b:g})" for a, b in zip(dL_dy[0], w[0]))
print(f"dL/dx[0,0] = {terms} = {np.dot(dL_dy[0], w[0]):g}")

Now the full matrices with the formulas:

In [ ]:
dL_dx = dL_dy @ w.T
dL_dw = x.T @ dL_dy
print("dL/dx = dL/dy @ w.T =\n", dL_dx, "  shape", dL_dx.shape, "(same as x)")
print("dL/dw = x.T @ dL/dy =\n", dL_dw, "  shape", dL_dw.shape, "(same as w)")

**Two independent checks.** (1) At this tiny size we *can* build the 4-D Jacobian and contract it; (2) a numerical check, using the loss $L = \sum(\frac{\partial L}{\partial y}\odot y)$, whose gradient with respect to $y$ is exactly our given $\frac{\partial L}{\partial y}$.

In [ ]:
J4 = np.zeros((2, 3, 2, 4))                                   # dy[k, m] / dx[n, d]
for n in range(2):
    for d in range(3):
        J4[n, d, n, :] = w[d, :]                              # = w[d, m] when k == n, else 0
print("explicit Jacobian gives the same dL/dx:", np.allclose(np.einsum("ndkm,km->nd", J4, dL_dy), dL_dx))

L_of = lambda x_, w_: np.sum(dL_dy * (x_ @ w_))
numerical = np.zeros_like(x)
for n in range(2):
    for d in range(3):
        x_plus = x.copy(); x_plus[n, d] += 1e-6
        numerical[n, d] = (L_of(x_plus, w) - L_of(x, w)) / 1e-6
print("numerical dL/dx matches:", np.allclose(numerical, dL_dx, atol=1e-4))

---
## 7. Putting it together: a network from modular layers

Every **layer** implements:
* `forward(x)`: compute the output, **saving** what backward will need;
* `backward(dout)`: given the upstream gradient, store gradients for its own parameters and **return the gradient for its input**.

A **network** is a list of layers: forward in order, backward in reverse. Adding a layer or changing the loss needs **no new derivations**. That's the modularity the "derive on paper" approach lacked.

**The three layers:**

In [ ]:
class Linear:
    # y = x W + b.  x: (N, D) -> y: (N, M).
    def __init__(self, D, M, rng):
        self.params = {"W": rng.normal(0, math.sqrt(2 / D), (D, M)), "b": np.zeros(M)}   # He initialization
        self.grads = {}
    def forward(self, x):
        self.x = x                                              # saved: dL/dW needs it
        return x @ self.params["W"] + self.params["b"]
    def backward(self, dout):
        self.grads = {"W": self.x.T @ dout, "b": dout.sum(0)}   # dL/dW = x^T dL/dy (Sec. 6)
        return dout @ self.params["W"].T                        # dL/dx = dL/dy W^T

class ReLU:
    # y = max(0, x), elementwise. No parameters.
    params, grads = {}, {}
    def forward(self, x):
        self.mask = x > 0                                       # saved: which inputs were positive
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask                                 # the implicit Jacobian of Sec. 5.1

class SoftmaxCrossEntropy:
    # Average softmax loss of scores (N, C) with labels y. backward() returns dL/dscores.
    def forward(self, scores, y):
        S = scores - scores.max(1, keepdims=True)
        P = np.exp(S); P /= P.sum(1, keepdims=True)
        self.P, self.y = P, y                                   # saved for backward
        return -np.log(P[np.arange(len(y)), y]).mean()
    def backward(self):
        d = self.P.copy(); d[np.arange(len(self.y)), self.y] -= 1
        return d / len(self.y)                                  # probabilities - one-hot, averaged

**The network**: `loss(X, y)` does the forward pass through every layer, then the backward pass through them in reverse.

In [ ]:
class Net:
    # A fully connected network: Linear -> ReLU -> ... -> Linear, with a softmax loss.
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed); self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], rng))
            if i < len(sizes) - 2:
                self.layers.append(ReLU())                      # no ReLU after the last layer
        self.loss_layer = SoftmaxCrossEntropy()

    def scores(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def loss(self, X, y):
        L = self.loss_layer.forward(self.scores(X), y)
        d = self.loss_layer.backward()
        for layer in reversed(self.layers):                     # backward, in reverse order
            d = layer.backward(d)
        return L

**Gradient check** a 64 → 50 → 30 → 10 network on 20 digits:

In [ ]:
digits = load_digits(); Xd = digits.data / 16.0 - 0.3; yd = digits.target
net = Net([64, 50, 30, 10], seed=1)
net.loss(Xd[:20], yd[:20])
check_rng = np.random.default_rng(2); worst = 0.0
for layer in net.layers:
    for k, P in layer.params.items():
        for _ in range(5):
            ix = tuple(check_rng.integers(0, s) for s in P.shape); old = P[ix]; g = layer.grads[k][ix]
            P[ix] = old + 1e-5; lp = net.loss(Xd[:20], yd[:20])
            P[ix] = old - 1e-5; lm = net.loss(Xd[:20], yd[:20])
            P[ix] = old; net.loss(Xd[:20], yd[:20])             # restore the weight and the stored gradients
            num = (lp - lm) / 2e-5
            worst = max(worst, abs(num - g) / max(abs(num), abs(g), 1e-10))
print(f"3-layer net built from modules: max relative gradient error {worst:.1e}")

**Train it with SGD** on the digits:

In [ ]:
order = check_rng.permutation(len(Xd)); tr_, te_ = order[:1300], order[1300:]
for step in range(2000):
    idx = check_rng.choice(tr_, 64)
    net.loss(Xd[idx], yd[idx])                                  # forward + backward
    for layer in net.layers:
        for k in layer.params:
            layer.params[k] -= 0.1 * layer.grads[k]             # gradient-descent update
print(f"digits test accuracy: {np.mean(net.scores(Xd[te_]).argmax(1) == yd[te_]):.1%}")

---
## 8. 🔎 Forward-mode vs. reverse-mode differentiation

Backprop is **reverse-mode** automatic differentiation: one backward sweep gives the derivative of **one output** (the scalar loss) with respect to **all inputs**. The alternative, **forward mode**, carries derivatives **along with** the forward computation, using **dual numbers** $a + b\varepsilon$ with the rule $\varepsilon^2 = 0$. Then $f(a + \varepsilon) = f(a) + f'(a)\,\varepsilon$: the $\varepsilon$ part carries the derivative. Tiny example: for $f(x) = x^2 + 3x$ at $x = 2$, $(2 + \varepsilon)^2 + 3(2 + \varepsilon) = 10 + 7\varepsilon$, and indeed $f'(2) = 2\cdot2 + 3 = 7$.

In [ ]:
class Dual:
    # a + b*eps with eps^2 = 0. The b part carries the derivative forward.
    def __init__(self, a, b=0.0): self.a, self.b = a, b
    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o); return Dual(self.a + o.a, self.b + o.b)
    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o); return Dual(self.a * o.a, self.a * o.b + self.b * o.a)   # product rule
    __radd__, __rmul__ = __add__, __mul__

result = Dual(2.0, 1.0) * Dual(2.0, 1.0) + 3 * Dual(2.0, 1.0)      # f(x) = x^2 + 3x at x = 2, seeded with dx = 1
print(f"f(2) = {result.a}, f'(2) = {result.b}")

**The catch:** one forward-mode pass gives the derivative with respect to **one input direction**. For $f:\mathbb{R}^N\to\mathbb{R}$ with $N$ parameters, that's **N passes**, while reverse mode needs **one** forward and one backward pass. Measured for $N = 300$:

In [ ]:
def f_many(xs):
    # A scalar function of N inputs: sum of 0.1*(i % 7) * x_i^2 + 0.5 * x_i. Works with floats, Duals, or Values.
    total = 0.0
    for i, xi in enumerate(xs):
        total = total + xi * xi * (0.1 * (i % 7)) + xi * 0.5
    return total

N_in = 300; x_vals = np.random.default_rng(0).normal(size=N_in)
t0 = time.time()
grad_forward = [f_many([Dual(v, 1.0 if j == i else 0.0) for j, v in enumerate(x_vals)]).b for i in range(N_in)]   # one pass per input
t_fwd = time.time() - t0
t0 = time.time()
xs_v = [Value(v) for v in x_vals]; out = f_many(xs_v); out.backward(); grad_reverse = [v.grad for v in xs_v]
t_rev = time.time() - t0
print(f"N = {N_in}: forward mode {N_in} passes ({t_fwd:.2f}s);  reverse mode 1 forward + 1 backward ({t_rev:.3f}s)")
print("same gradient:", np.allclose(grad_forward, grad_reverse))

**What to notice:** both modes give the same gradient, but for "one loss, many parameters" (exactly our situation) reverse mode is the efficient direction. (Forward mode wins for "few inputs, many outputs", and is the efficient way to compute **Jacobian-vector products**; JAX exposes both, as `jvp` and `vjp`.)

---
## 9. Higher-order derivatives

Backprop can also give **second** derivatives. The **Hessian** $\frac{\partial^2L}{\partial w^2}$ ($N\times N$) is too big to build for real networks (Lecture 4: $O(N^2)$), but a **Hessian-vector product** $Hv$ is cheap: it's the gradient of the scalar $(\nabla L)\cdot v$, so you can **backpropagate twice**, at about twice the cost of one gradient ("double backprop": PyTorch's `create_graph=True`). Uses: second-order optimizers, measuring curvature, and **penalties on the size of the gradient itself** (e.g., WGAN-GP, Lecture 20), $R = \|\partial L/\partial x\|^2$, whose own gradient needs second derivatives.

**Check on logistic regression:** an $Hv$ computed from just two gradients (a finite difference of gradients along $v$, standing in for the double backward pass) vs. the explicit Hessian times $v$:

In [ ]:
hv_rng = np.random.default_rng(3)
Xl = hv_rng.normal(size=(200, 30)); yl = (Xl @ hv_rng.normal(size=30) > 0).astype(float)

def grad_logistic(w):
    # Gradient of the average logistic loss.
    p = 1 / (1 + np.exp(-Xl @ w)); return Xl.T @ (p - yl) / len(yl)

def hessian_logistic(w):
    # Explicit 30 x 30 Hessian of the average logistic loss.
    p = 1 / (1 + np.exp(-Xl @ w)); return (Xl.T * (p * (1 - p))) @ Xl / len(yl)

w_ = hv_rng.normal(size=30) * 0.1; v = hv_rng.normal(size=30)
hvp_cheap = (grad_logistic(w_ + 1e-5 * v) - grad_logistic(w_ - 1e-5 * v)) / 2e-5    # two gradients, O(N) memory
print(f"Hv from two gradients vs explicit Hessian @ v: max difference {np.abs(hvp_cheap - hessian_logistic(w_) @ v).max():.1e}")

---
## 10. 🔎 The memory cost of backprop

To compute $\frac{\partial L}{\partial W} = x^\top\frac{\partial L}{\partial y}$, a linear layer needs its **input** $x$ from the forward pass; ReLU needs its mask; and so on. So backprop **stores every layer's activations** until the backward pass reaches it, and training memory grows with depth × batch size.
* For **convolutional networks**, which apply few weights at many positions, **activations dominate** memory.
* **Gradient checkpointing** (Chen et al. 2016) trades compute for memory: store only some activations and recompute the rest during the backward pass.
* At inference nothing needs storing, which is why `torch.no_grad()` saves so much memory.

**VGG-16 (Lecture 8)**: adding up the outputs of its 13 conv layers per image:

In [ ]:
vgg_stages = [(224, 64, 2), (112, 128, 2), (56, 256, 3), (28, 512, 3), (14, 512, 3)]   # (size, channels, conv layers)
activations_per_image = sum(size * size * ch * n for size, ch, n in vgg_stages)       # conv outputs only
vgg_weights = 138_357_544                                                             # VGG-16's parameter count
print(f"VGG-16 weights: {vgg_weights * 4 / 2 ** 20:,.0f} MB;  conv activations per image: {activations_per_image / 1e6:.1f}M numbers "
      f"= {activations_per_image * 4 / 2 ** 20:.0f} MB")
for batch in [1, 32, 128]:
    print(f"  batch {batch:4d}: activations ~{batch * activations_per_image * 4 / 2 ** 30:6.2f} GB  vs. weights {vgg_weights * 4 / 2 ** 30:.2f} GB")
widths = [3072] + [4096] * 8 + [10]
mlp_params = sum(a * b + b for a, b in zip(widths[:-1], widths[1:]))
print(f"contrast, a 9-layer MLP: weights {mlp_params * 4 / 2 ** 20:,.0f} MB;  activations at batch 128 {128 * sum(widths) * 4 / 2 ** 20:.0f} MB")

**What to notice:** VGG-16 needs several times more memory for activations than for weights at an ordinary batch size; a fully connected network is the opposite.

---
## 11. PyTorch autograd (run in Colab)
PyTorch builds the computational graph **as your code runs** ("define by run") and backpropagates with `.backward()`. Below: the lecture's examples, a custom `autograd.Function` with the same forward/backward/`ctx.save_for_backward` structure as §4.2, a gradient check, and double backprop.

In [ ]:
# [PyTorch] Needs: pip install torch.
import torch
x, y, z = (torch.tensor(v, requires_grad=True) for v in (-2.0, 5.0, -4.0))
f = (x + y) * z; f.backward()
print("(x + y) z:", f.item(), "grads", x.grad.item(), y.grad.item(), z.grad.item())

class MySigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        out = 1 / (1 + torch.exp(-x))
        ctx.save_for_backward(out)            # save what backward needs
        return out
    @staticmethod
    def backward(ctx, grad_out):
        (out,) = ctx.saved_tensors
        return grad_out * (1 - out) * out     # upstream x local

w = torch.tensor([2.0, -3.0, -3.0], requires_grad=True); xv = torch.tensor([-1.0, -2.0, 1.0])
out = MySigmoid.apply((w * xv).sum()); out.backward()
print("sigmoid neuron grads (w0, w1, w2):", w.grad)
print("gradcheck(MySigmoid):", torch.autograd.gradcheck(MySigmoid.apply, (torch.randn(5, dtype=torch.float64, requires_grad=True),)))

wv = torch.randn(10, requires_grad=True); v = torch.randn(10)
loss = (wv ** 3).sum()                                          # Hessian = diag(6 w)
(g,) = torch.autograd.grad(loss, wv, create_graph=True)         # keep the graph of the gradient computation
(hvp,) = torch.autograd.grad(g @ v, wv)                          # backprop through it again
print("double-backprop HVP matches 6 * w * v:", torch.allclose(hvp, 6 * wv.detach() * v))

---
## 12. Summary

**[CS231n 2026]'s summary:** fully connected networks are stacks of linear functions and nonlinear activations, with much more power than linear classifiers; **backpropagation = recursive application of the chain rule along a computational graph**; implementations keep a graph whose nodes implement **forward() / backward()**; **forward** computes the result and saves intermediates needed for the gradient; **backward** applies the chain rule to get the loss's gradient with respect to the inputs.

| Topic | Takeaway |
|---|---|
| Computational graph | Break the computation into simple nodes with easy local derivatives |
| Chain rule | Downstream gradient = upstream gradient × local gradient |
| Patterns | add distributes, multiply swaps, copy adds, max routes |
| Implementation | Flat code (forward lines, then reverse gradient lines) or forward/backward objects; autograd = topological sort + upstream × local |
| Vectors | Local gradients are Jacobians, but never build them: use implicit products |
| Matrices | $y = xw$: $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}w^\top$, $\frac{\partial L}{\partial w} = x^\top\frac{\partial L}{\partial y}$ (make the shapes match) |
| 🔎 Reverse mode | One backward pass for all N parameters (forward mode would need N) |
| Higher order | Hessian-vector products by backpropagating twice |
| 🔎 Memory | Backprop stores activations; checkpointing trades compute for memory |

### Check yourself
<details><summary>1. In (x + y) z with x = −2, y = 5, z = −4, why is ∂f/∂x = −4?</summary>

The upstream gradient at q = x + y is ∂f/∂q = z = −4, and the add node's local gradient is 1, so ∂f/∂x = −4 × 1.
</details>

<details><summary>2. A weight is used in two places in a graph. How do you combine its gradients?</summary>

Add them (the copy rule). That's why autograd engines accumulate with +=.
</details>

<details><summary>3. For y = xw with x: N×D and w: D×M, what are dL/dx and dL/dw?</summary>

dL/dx = (dL/dy) wᵀ (N×M times M×D = N×D); dL/dw = xᵀ (dL/dy) (D×N times N×M = D×M).
</details>

<details><summary>4. Why don't we build the Jacobian of a ReLU layer?</summary>

It's a diagonal D×D matrix: building it wastes O(D²) memory, while multiplying by it is just masking the upstream gradient.
</details>

<details><summary>5. Why is reverse mode preferred over forward mode for training?</summary>

We need the gradient of one scalar (the loss) with respect to millions of parameters: reverse mode gets all of them in one backward pass; forward mode needs one pass per parameter.
</details>

### Exercises
1. Add `tanh`, `sigmoid`, and `log` to the `Value` engine, and train a tiny 2-layer network on XOR with it.
2. Derive the backward pass of **softmax on its own** (Jacobian $\text{diag}(p) - pp^\top$) and verify it with the engine.
3. Write **batch normalization**'s backward pass (Lecture 7) node by node as a computational graph, then simplify.
4. Implement a `Linear` layer whose backward uses the explicit 4-D Jacobian, and time it against the implicit version for D = M = 256.
5. Compute §9's Hessian-vector product with **double backprop** in the `Value` engine (hint: make each `_backward` itself build graph nodes).

### References
* CS231n course notes, *Backpropagation, Intuitions* (cs231n.github.io/optimization-2); linear-backprop handout (cs231n.stanford.edu/handouts/linear-backprop.pdf).
* Rumelhart, Hinton & Williams 1986; Linnainmaa 1970; Werbos 1974.
* Baydin et al. 2018 (*Automatic Differentiation in Machine Learning: a Survey*); Karpathy, *micrograd*; Paszke et al. 2019 (PyTorch).
* Pearlmutter 1994 (Hessian-vector products); Chen et al. 2016 (gradient checkpointing); Gulrajani et al. 2017 (WGAN-GP).

**Next:** Lecture 7, Convolutional Networks.